# OpenL3 + SVM

**Модель:** OpenL3 эмбеддинги → агрегация mean+max → SVM RBF

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import openl3
import soundfile as sf
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score
from collections import Counter

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate, evaluate_cv_folds, print_cv_summary
from shared.data_utils import build_feature_matrix, get_cv_folds
from shared.results_utils import save_result_csv

import warnings
warnings.filterwarnings("ignore")

In [ ]:
def extract_openl3(path: str) -> np.ndarray:
    """OpenL3 audio-audio → mean+max (1024-dim)."""
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    emb, ts = openl3.get_audio_embedding(
        audio, sr,
        content_type="music",
        embedding_size=512,
        hop_size=0.5,
        verbose=False,
    )
    return np.concatenate([emb.mean(axis=0), emb.max(axis=0)]).astype(np.float32)

In [ ]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)} good: {np.sum(labels_trainval==0)}, bad: {np.sum(labels_trainval==1)}")
print(f"Test:      {len(paths_test)}  good: {np.sum(labels_test==0)},  bad: {np.sum(labels_test==1)}")

## 5-Fold CV (для подбора гиперпараметров)

In [ ]:
print("Извлечение эмбеддингов (train+val)...")
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_openl3, n_jobs=4)
print("Извлечение эмбеддингов (test)...")
X_embeds_test = build_feature_matrix(paths_test, extract_openl3, n_jobs=4)

path_to_idx = {p: i for i, p in enumerate(paths_trainval)}

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")

param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

fold_results = []
all_best_params = []
t0 = time.perf_counter()

for paths_tr, paths_val, labels_tr, labels_val, letters_tr, letters_val, fold in \
        get_cv_folds(paths_trainval, labels_trainval, letters_trainval):

    idx_tr  = [path_to_idx[p] for p in paths_tr]
    idx_val = [path_to_idx[p] for p in paths_val]
    X_tr  = np.hstack([X_embeds_trainval[idx_tr],  letters_tr])
    X_val = np.hstack([X_embeds_trainval[idx_val], letters_val])

    grid = GridSearchCV(pipeline, param_grid, cv=3,
                        scoring=f1_bad_scorer, refit=True, n_jobs=-1)
    grid.fit(X_tr, labels_tr)
    clf = grid.best_estimator_
    all_best_params.append(grid.best_params_)
    y_proba_val = clf.predict_proba(X_val)[:, config.CLASS_BAD]
    thr = find_optimal_threshold(labels_val, y_proba_val)
    metrics = evaluate(labels_val, y_proba_val, threshold=thr, verbose=False)
    print(f"Фолд {fold+1}/{config.CV_N_SPLITS}, best_params={grid.best_params_}, threshold={thr}")
    fold_results.append(metrics)

train_time_sec = time.perf_counter() - t0
cv_agg = evaluate_cv_folds(fold_results)
print_cv_summary(cv_agg)

## Финальная модель: обучение на train+val, оценка на test

In [ ]:
X_trainval = np.hstack([X_embeds_trainval, letters_trainval])
X_test     = np.hstack([X_embeds_test,     letters_test])

best_params = dict(
    Counter(tuple(sorted(p.items())) for p in all_best_params).most_common(1)[0][0]
)
print(f"Лучшие параметры из CV: {best_params}")
pipeline.set_params(**best_params)
pipeline.fit(X_trainval, labels_trainval)
best_clf = pipeline

cv_threshold = cv_agg["threshold_mean"]
y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=cv_threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= cv_threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_openl3_svm",
    experiment_name="OpenL3 + SVM",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=1024,
    embed_dim_note="OpenL3 512-dim × 2 (mean+max)",
    notes=f"5-fold CV + test | best_params={best_params} | cv_thr={cv_threshold:.2f}",
    train_time_sec=train_time_sec,
)